In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns
import pandas as pd
from collections import Counter
from torchvision import models
import torch.nn.functional as F
from pathlib import Path
import json
import logging
from typing import Dict, List, Tuple, Optional, Any
import warnings
from tqdm import tqdm
import time
from torch.cuda.amp import GradScaler, autocast
import random
import multiprocessing

# Configuração de logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Suprimir warnings desnecessários
warnings.filterwarnings('ignore', category=UserWarning)

# Configuração de dispositivo com otimizações
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Dispositivo utilizado: {device}")

if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name()}")
    logger.info(f"Memória GPU disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Otimizações CUDA
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

# Configuração de sementes para reprodutibilidade
def set_seed(seed: int = 42):
    """Define sementes para reprodutibilidade"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

def get_optimized_transforms(phase: str = 'train', use_albumentations: bool = True):
    """Transformações otimizadas com fallback para torchvision"""
    
    if use_albumentations:
        try:
            import albumentations as A
            from albumentations.pytorch import ToTensorV2
            
            if phase == 'train':
                return A.Compose([
                    A.Resize(256, 256),
                    A.RandomResizedCrop(224, 224, scale=(0.9, 1.0)),
                    A.HorizontalFlip(p=0.5),
                    A.Rotate(limit=10, p=0.5),
                    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
                    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
                    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
                    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                    ToTensorV2(),
                ])
            else:
                return A.Compose([
                    A.Resize(224, 224),
                    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                    ToTensorV2(),
                ])
        except ImportError:
            logger.warning("Albumentations não disponível. Usando torchvision transforms.")
            use_albumentations = False
    
    # Fallback para torchvision
    if phase == 'train':
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

class OptimizedCOVIDDataset(Dataset):
    """Dataset otimizado para COVID-19 com cache e validações melhoradas"""
    
    def __init__(self, dataset_root: str, split: str = 'train', transform=None, cache_images: bool = False):
        self.dataset_root = Path(dataset_root)
        self.split = split
        self.transform = transform
        self.cache_images = cache_images
        self.image_cache = {}
        self.samples = []
        
        # Classes definidas
        self.class_names = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']
        self.class_to_idx = {class_name: idx for idx, class_name in enumerate(self.class_names)}
        
        self._load_samples()
        self._validate_dataset()
        
        if self.cache_images and len(self.samples) > 0:
            self._cache_images()

    def _load_samples(self):
        """Carrega amostras com validação otimizada"""
        split_path = self.dataset_root / self.split
        
        if not split_path.exists():
            raise FileNotFoundError(f"Caminho não encontrado: {split_path}")
        
        logger.info(f"Carregando dados de: {split_path}")
        
        # Extensões de imagem suportadas
        valid_extensions = {'.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG'}
        
        for class_name in self.class_names:
            class_path = split_path / class_name
            
            if not class_path.exists():
                logger.warning(f"Classe {class_name} não encontrada em {class_path}")
                continue
            
            # Busca otimizada usando glob
            images_found = 0
            for img_path in class_path.iterdir():
                if img_path.suffix in valid_extensions and img_path.is_file():
                    # Verificação rápida se a imagem pode ser aberta
                    try:
                        with Image.open(img_path) as img:
                            img.verify()  # Verificação rápida
                        self.samples.append((str(img_path), self.class_to_idx[class_name]))
                        images_found += 1
                    except Exception as e:
                        logger.warning(f"Imagem corrompida ignorada: {img_path} - {e}")
            
            logger.info(f"  {class_name}: {images_found} imagens válidas")

    def _validate_dataset(self):
        """Valida o dataset carregado"""
        if not self.samples:
            raise ValueError("Nenhuma amostra válida encontrada!")
        
        # Verificar distribuição das classes
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        
        logger.info(f"\nDistribuição das classes ({self.split}):")
        total = len(self.samples)
        
        for class_idx, class_name in enumerate(self.class_names):
            count = class_counts.get(class_idx, 0)
            percentage = (count / total) * 100 if total > 0 else 0
            logger.info(f"  {class_name}: {count} amostras ({percentage:.1f}%)")
        
        # Verificar desbalanceamento extremo
        if class_counts:
            max_count = max(class_counts.values())
            min_count = min(class_counts.values())
            if max_count / min_count > 10:
                logger.warning(f"Dataset muito desbalanceado! Razão: {max_count/min_count:.1f}")

    def _cache_images(self):
        """Cache de imagens para acelerar o treinamento"""
        logger.info("Fazendo cache das imagens...")
        for idx in tqdm(range(len(self.samples)), desc="Caching images"):
            img_path, _ = self.samples[idx]
            try:
                image = Image.open(img_path).convert("RGB")
                self.image_cache[img_path] = image
            except Exception as e:
                logger.warning(f"Erro ao cachear imagem {img_path}: {e}")

    def get_class_weights(self) -> torch.Tensor:
        """Calcula pesos das classes usando estratégia balanceada"""
        if not self.samples:
            return torch.ones(len(self.class_names))
        
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        total_samples = len(self.samples)
        
        # Usar 'balanced' strategy: n_samples / (n_classes * n_samples_per_class)
        weights = []
        for i in range(len(self.class_names)):
            if i in class_counts and class_counts[i] > 0:
                weight = total_samples / (len(self.class_names) * class_counts[i])
                weights.append(weight)
            else:
                weights.append(1.0)
        
        weights_tensor = torch.FloatTensor(weights)
        logger.info(f"Pesos das classes: {weights_tensor}")
        return weights_tensor

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        if idx >= len(self.samples):
            raise IndexError(f"Index {idx} out of range for dataset with {len(self.samples)} samples")
        
        img_path, label = self.samples[idx]
        
        try:
            # Usar cache se disponível
            if img_path in self.image_cache:
                image = self.image_cache[img_path].copy()
            else:
                image = Image.open(img_path).convert("RGB")
            
            if self.transform:
                # Para Albumentations
                if hasattr(self.transform, 'processors'):
                    image_np = np.array(image)
                    transformed = self.transform(image=image_np)
                    image = transformed['image']
                else:
                    # Para torchvision transforms
                    image = self.transform(image)
            else:
                # Transformação básica
                image = transforms.ToTensor()(image)
                image = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(image)

            return image, label
            
        except Exception as e:
            logger.warning(f"Erro ao carregar imagem {img_path}: {e}")
            # Retornar imagem preta em caso de erro
            dummy_image = torch.zeros(3, 224, 224)
            return dummy_image, label

class OptimizedCOVIDClassifier(nn.Module):
    """Classificador otimizado com arquitetura melhorada"""
    
    def __init__(self, num_classes: int = 4, pretrained: bool = True, 
                 dropout_rate: float = 0.3, architecture: str = 'efficientnet_b3'):
        super(OptimizedCOVIDClassifier, self).__init__()
        
        self.num_classes = num_classes
        self.architecture = architecture
        
        # Backbone otimizado
        if architecture == 'efficientnet_b3':
            try:
                from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
                self.backbone = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT if pretrained else None)
                num_features = self.backbone.classifier[1].in_features
                self.backbone.classifier = nn.Identity()
            except ImportError:
                logger.warning("EfficientNet não disponível. Usando ResNet50.")
                architecture = 'resnet50'
        
        if architecture == 'resnet50':
            self.backbone = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        
        # Classificador melhorado com regularização
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1) if architecture == 'efficientnet_b3' else nn.Identity(),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate * 0.25),
            nn.Linear(256, num_classes)
        )
        
        self._initialize_weights()

    def _initialize_weights(self):
        """Inicialização otimizada dos pesos"""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.classifier(features)

class OptimizedModelTrainer:
    """Trainer otimizado com mixed precision e métricas avançadas"""
    
    def __init__(self, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                 criterion: nn.Module, optimizer: optim.Optimizer, scheduler=None,
                 device: str = 'cpu', class_names: List[str] = None, use_amp: bool = True):
        
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.class_names = class_names or [f'Class {i}' for i in range(4)]
        self.use_amp = use_amp and torch.cuda.is_available()
        
        # Mixed precision scaler
        self.scaler = GradScaler() if self.use_amp else None
        
        # Histórico otimizado
        self.history = {
            'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [],
            'per_class_metrics': [], 'learning_rates': []
        }
        
        logger.info(f"Trainer configurado com Mixed Precision: {self.use_amp}")

    def train_epoch(self) -> Tuple[float, float, Dict[str, float]]:
        """Treina uma época com otimizações"""
        self.model.train()
        running_loss = 0.0
        all_predictions = []
        all_labels = []
        
        # Progress bar
        pbar = tqdm(self.train_loader, desc="Training", leave=False)
        
        for batch_idx, (images, labels) in enumerate(pbar):
            images, labels = images.to(self.device, non_blocking=True), labels.to(self.device, non_blocking=True)
            
            self.optimizer.zero_grad()
            
            if self.use_amp:
                with autocast():
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Atualizar progress bar
            pbar.set_postfix({'Loss': f'{loss.item():.4f}'})
        
        epoch_loss = running_loss / len(self.train_loader)
        epoch_acc = np.mean(np.array(all_predictions) == np.array(all_labels))
        class_acc = self._calculate_per_class_accuracy(all_labels, all_predictions)
        
        return epoch_loss, epoch_acc, class_acc
    
    def validate_epoch(self) -> Tuple[float, float, Dict[str, float]]:
        """Valida uma época com otimizações"""
        self.model.eval()
        running_loss = 0.0
        all_predictions = []
        all_labels = []
        
        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc="Validating", leave=False)
            for images, labels in pbar:
                images, labels = images.to(self.device, non_blocking=True), labels.to(self.device, non_blocking=True)
                
                if self.use_amp:
                    with autocast():
                        outputs = self.model(images)
                        loss = self.criterion(outputs, labels)
                else:
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                all_predictions.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
                pbar.set_postfix({'Loss': f'{loss.item():.4f}'})
        
        epoch_loss = running_loss / len(self.val_loader)
        epoch_acc = np.mean(np.array(all_predictions) == np.array(all_labels))
        class_acc = self._calculate_per_class_accuracy(all_labels, all_predictions)
        
        return epoch_loss, epoch_acc, class_acc
    
    def _calculate_per_class_accuracy(self, labels: List[int], predictions: List[int]) -> Dict[str, float]:
        """Calcula acurácia por classe otimizada"""
        labels_np = np.array(labels)
        predictions_np = np.array(predictions)
        
        class_acc = {}
        for i, class_name in enumerate(self.class_names):
            class_mask = labels_np == i
            if np.sum(class_mask) > 0:
                class_predictions = predictions_np[class_mask]
                class_labels = labels_np[class_mask]
                class_acc[class_name] = np.mean(class_predictions == class_labels)
            else:
                class_acc[class_name] = 0.0
        return class_acc
    
    def train(self, num_epochs: int, early_stopping_patience: int = 10) -> float:
        """Treinamento otimizado com early stopping"""
        best_val_acc = 0.0
        patience_counter = 0
        start_time = time.time()
        
        logger.info(f"Iniciando treinamento por {num_epochs} épocas...")
        logger.info("-" * 80)
        
        for epoch in range(num_epochs):
            epoch_start = time.time()
            logger.info(f'Época {epoch+1}/{num_epochs}')
            
            # Treinamento
            train_loss, train_acc, train_class_acc = self.train_epoch()
            
            # Validação
            val_loss, val_acc, val_class_acc = self.validate_epoch()
            
            # Scheduler
            current_lr = self.optimizer.param_groups[0]['lr']
            if self.scheduler:
                if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(val_loss)
                else:
                    self.scheduler.step()
                
                new_lr = self.optimizer.param_groups[0]['lr']
                if new_lr != current_lr:
                    logger.info(f'Learning rate: {current_lr:.6f} -> {new_lr:.6f}')
            
            # Salvar histórico
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            self.history['learning_rates'].append(current_lr)
            self.history['per_class_metrics'].append({
                'epoch': epoch,
                'train_class_acc': train_class_acc,
                'val_class_acc': val_class_acc
            })
            
            epoch_time = time.time() - epoch_start
            logger.info(f'Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}')
            logger.info(f'Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')
            logger.info(f'Tempo da época: {epoch_time:.2f}s')
            
            # Mostrar acurácia por classe
            logger.info("Acurácia por classe (Validação):")
            for class_name, acc in val_class_acc.items():
                logger.info(f"  {class_name}: {acc:.4f}")
            
            # Early stopping com salvamento do melhor modelo
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                
                # Salvar melhor modelo
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'scheduler_state_dict': self.scheduler.state_dict() if self.scheduler else None,
                    'val_acc': val_acc,
                    'val_loss': val_loss,
                    'class_acc': val_class_acc,
                    'history': self.history
                }, 'best_covid_model_optimized.pth')
                
                logger.info(f'✓ Melhor modelo salvo! Val Acc: {val_acc:.4f}')
            else:
                patience_counter += 1
            
            if patience_counter >= early_stopping_patience:
                logger.info(f'Early stopping após {early_stopping_patience} épocas sem melhoria')
                break
            
            logger.info("-" * 80)
        
        total_time = time.time() - start_time
        logger.info(f'Treinamento concluído em {total_time:.2f}s')
        
        return best_val_acc

    def plot_training_history(self, save_path: str = 'training_history_optimized.png'):
        """Plot otimizado do histórico de treinamento"""
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        epochs = range(1, len(self.history['train_loss']) + 1)
        
        # Loss
        axes[0, 0].plot(epochs, self.history['train_loss'], 'b-', label='Train Loss', linewidth=2)
        axes[0, 0].plot(epochs, self.history['val_loss'], 'r-', label='Val Loss', linewidth=2)
        axes[0, 0].set_title('Model Loss', fontsize=14, fontweight='bold')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Accuracy
        axes[0, 1].plot(epochs, self.history['train_acc'], 'b-', label='Train Acc', linewidth=2)
        axes[0, 1].plot(epochs, self.history['val_acc'], 'r-', label='Val Acc', linewidth=2)
        axes[0, 1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # Learning Rate
        axes[0, 2].plot(epochs, self.history['learning_rates'], 'g-', linewidth=2)
        axes[0, 2].set_title('Learning Rate', fontsize=14, fontweight='bold')
        axes[0, 2].set_xlabel('Epoch')
        axes[0, 2].set_ylabel('Learning Rate')
        axes[0, 2].set_yscale('log')
        axes[0, 2].grid(True, alpha=0.3)
        
        # Acurácia por classe
        if self.history['per_class_metrics']:
            for class_name in self.class_names:
                class_accs = [metric['val_class_acc'].get(class_name, 0) 
                             for metric in self.history['per_class_metrics']]
                axes[1, 0].plot(epochs, class_accs, label=f'{class_name}', 
                               linewidth=2, marker='o', markersize=3)
            
            axes[1, 0].set_title('Validation Accuracy by Class', fontsize=14, fontweight='bold')
            axes[1, 0].set_xlabel('Epoch')
            axes[1, 0].set_ylabel('Class Accuracy')
            axes[1, 0].legend()
            axes[1, 0].grid(True, alpha=0.3)
# Continuação da função plot_training_history
        
        # Overfitting detection
        gap = np.array(self.history['train_acc']) - np.array(self.history['val_acc'])
        axes[1, 1].plot(epochs, gap, 'orange', linewidth=2, label='Train-Val Gap')
        axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
        axes[1, 1].set_title('Overfitting Detection', fontsize=14, fontweight='bold')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Accuracy Gap')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        # Metrics summary
        final_train_acc = self.history['train_acc'][-1] if self.history['train_acc'] else 0
        final_val_acc = self.history['val_acc'][-1] if self.history['val_acc'] else 0
        best_val_acc = max(self.history['val_acc']) if self.history['val_acc'] else 0
        
        summary_text = f"""
        Final Training Accuracy: {final_train_acc:.4f}
        Final Validation Accuracy: {final_val_acc:.4f}
        Best Validation Accuracy: {best_val_acc:.4f}
        Total Epochs: {len(epochs)}
        """
        
        axes[1, 2].text(0.1, 0.5, summary_text, fontsize=12, 
                        verticalalignment='center', transform=axes[1, 2].transAxes,
                        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.5))
        axes[1, 2].set_title('Training Summary', fontsize=14, fontweight='bold')
        axes[1, 2].axis('off')
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        logger.info(f"Histórico de treinamento salvo em: {save_path}")

def create_optimized_dataloader(dataset: Dataset, batch_size: int = 32, 
                               is_train: bool = True, num_workers: int = None) -> DataLoader:
    """Cria DataLoader otimizado com configurações adequadas"""
    
    if num_workers is None:
        num_workers = min(multiprocessing.cpu_count(), 8)
    
    # Configurações otimizadas
    pin_memory = torch.cuda.is_available()
    persistent_workers = num_workers > 0
    
    if is_train and hasattr(dataset, 'get_class_weights'):
        # Weighted sampling para dados desbalanceados
        try:
            class_weights = dataset.get_class_weights()
            labels = [dataset.samples[i][1] for i in range(len(dataset))]
            sample_weights = [class_weights[label] for label in labels]
            sampler = WeightedRandomSampler(weights=sample_weights, 
                                          num_samples=len(sample_weights), 
                                          replacement=True)
            shuffle = False
        except Exception as e:
            logger.warning(f"Erro ao criar WeightedRandomSampler: {e}. Usando shuffle padrão.")
            sampler = None
            shuffle = True
    else:
        sampler = None
        shuffle = is_train
    
    dataloader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        drop_last=is_train,
        prefetch_factor=2 if num_workers > 0 else 2
    )
    
    logger.info(f"DataLoader criado - Batch size: {batch_size}, Workers: {num_workers}, "
                f"Pin memory: {pin_memory}, Sampler: {'Weighted' if sampler else 'None'}")
    
    return dataloader

def evaluate_model_optimized(model: nn.Module, test_loader: DataLoader, 
                           class_names: List[str], device: str) -> Dict[str, Any]:
    """Avaliação completa e otimizada do modelo"""
    
    logger.info("Iniciando avaliação do modelo...")
    model.eval()
    
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            outputs = model(images)
            probabilities = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    # Métricas detalhadas
    accuracy = np.mean(np.array(all_predictions) == np.array(all_labels))
    
    # Classification report
    report = classification_report(all_labels, all_predictions, 
                                 target_names=class_names, output_dict=True)
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    # ROC AUC (multiclass)
    try:
        all_probabilities_np = np.array(all_probabilities)
        roc_auc = roc_auc_score(all_labels, all_probabilities_np, 
                               multi_class='ovr', average='weighted')
    except Exception as e:
        logger.warning(f"Erro ao calcular ROC AUC: {e}")
        roc_auc = None
    
    # Preparar resultados
    results = {
        'accuracy': accuracy,
        'classification_report': report,
        'confusion_matrix': cm,
        'roc_auc': roc_auc,
        'predictions': all_predictions,
        'labels': all_labels,
        'probabilities': all_probabilities_np if 'all_probabilities_np' in locals() else all_probabilities
    }
    
    # Log dos resultados
    logger.info(f"Acurácia do modelo: {accuracy:.4f}")
    logger.info(f"ROC AUC: {roc_auc:.4f}" if roc_auc else "ROC AUC: N/A")
    
    # Exibir métricas por classe
    logger.info("\nMétricas por classe:")
    for class_name in class_names:
        if class_name in report:
            precision = report[class_name]['precision']
            recall = report[class_name]['recall']
            f1 = report[class_name]['f1-score']
            logger.info(f"  {class_name}: Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")
    
    # Plotar resultados
    _plot_confusion_matrix(cm, class_names)
    if roc_auc and len(class_names) <= 10:  # Evitar plots muito complexos
        _plot_roc_curves(all_labels, all_probabilities, class_names)
    
    return results

def _plot_confusion_matrix(cm: np.ndarray, class_names: List[str], 
                          save_path: str = 'confusion_matrix_optimized.png'):
    """Plot otimizado da matriz de confusão"""
    
    plt.figure(figsize=(10, 8))
    
    # Normalizar matriz de confusão
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot com seaborn
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'shrink': 0.8})
    
    plt.title('Normalized Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    # Adicionar valores absolutos como texto
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            plt.text(j + 0.5, i + 0.7, f'({cm[i, j]})', 
                    ha='center', va='center', fontsize=9, color='gray')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    logger.info(f"Matriz de confusão salva em: {save_path}")

def _plot_roc_curves(labels: List[int], probabilities: np.ndarray, 
                    class_names: List[str], save_path: str = 'roc_curves_optimized.png'):
    """Plot otimizado das curvas ROC"""
    
    from sklearn.metrics import roc_curve, auc
    from sklearn.preprocessing import label_binarize
    
    # Binarizar labels para multiclass ROC
    labels_bin = label_binarize(labels, classes=range(len(class_names)))
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.Set1(np.linspace(0, 1, len(class_names)))
    
    for i, (class_name, color) in enumerate(zip(class_names, colors)):
        if labels_bin.ndim > 1:
            y_true = labels_bin[:, i]
        else:
            y_true = (np.array(labels) == i).astype(int)
        
        y_score = probabilities[:, i]
        
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)
        
        plt.plot(fpr, tpr, color=color, lw=2, 
                label=f'{class_name} (AUC = {roc_auc:.3f})')
    
    # Linha diagonal
    plt.plot([0, 1], [0, 1], 'k--', lw=2, alpha=0.5)
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curves - Multi-class Classification', fontsize=16, fontweight='bold')
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    logger.info(f"Curvas ROC salvas em: {save_path}")

def save_model_info(model: nn.Module, history: Dict, results: Dict, 
                   save_path: str = 'model_info_optimized.json'):
    """Salva informações detalhadas do modelo"""
    
    model_info = {
        'model_architecture': str(model),
        'total_parameters': sum(p.numel() for p in model.parameters()),
        'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'model_size_mb': sum(p.numel() * p.element_size() for p in model.parameters()) / 1024 / 1024,
        'training_history': {
            'final_train_acc': history['train_acc'][-1] if history['train_acc'] else 0,
            'final_val_acc': history['val_acc'][-1] if history['val_acc'] else 0,
            'best_val_acc': max(history['val_acc']) if history['val_acc'] else 0,
            'total_epochs': len(history['train_acc']),
        },
        'test_results': {
            'accuracy': float(results['accuracy']),
            'roc_auc': float(results['roc_auc']) if results['roc_auc'] else None,
            'per_class_metrics': {
                class_name: {
                    'precision': float(metrics['precision']),
                    'recall': float(metrics['recall']),
                    'f1_score': float(metrics['f1-score']),
                    'support': int(metrics['support'])
                }
                for class_name, metrics in results['classification_report'].items()
                if isinstance(metrics, dict) and 'precision' in metrics
            }
        },
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'device_info': {
            'device': str(device),
            'cuda_available': torch.cuda.is_available(),
            'gpu_name': torch.cuda.get_device_name() if torch.cuda.is_available() else None
        }
    }
    
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(model_info, f, indent=2, ensure_ascii=False)
    
    logger.info(f"Informações do modelo salvas em: {save_path}")
    return model_info

def load_best_model(model: nn.Module, checkpoint_path: str = 'best_covid_model_optimized.pth') -> Dict:
    """Carrega o melhor modelo salvo"""
    
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint não encontrado: {checkpoint_path}")
    
    logger.info(f"Carregando modelo de: {checkpoint_path}")
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    logger.info(f"Modelo carregado - Época: {checkpoint['epoch']}, "
                f"Val Acc: {checkpoint['val_acc']:.4f}")
    
    return checkpoint

def predict_single_image(model: nn.Module, image_path: str, class_names: List[str],
                        device: str, transform=None) -> Dict[str, Any]:
    """Predição para uma única imagem com análise detalhada"""
    
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Imagem não encontrada: {image_path}")
    
    # Carregar e transformar imagem
    try:
        image = Image.open(image_path).convert('RGB')
        original_size = image.size
    except Exception as e:
        raise ValueError(f"Erro ao carregar imagem: {e}")
    
    if transform is None:
        transform = get_optimized_transforms(phase='val', use_albumentations=False)
    
    # Aplicar transformações
    if hasattr(transform, 'processors'):  # Albumentations
        image_np = np.array(image)
        transformed = transform(image=image_np)
        input_tensor = transformed['image'].unsqueeze(0)
    else:  # torchvision
        input_tensor = transform(image).unsqueeze(0)
    
    # Predição
    model.eval()
    with torch.no_grad():
        input_tensor = input_tensor.to(device)
        outputs = model(input_tensor)
        probabilities = F.softmax(outputs, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0, predicted_class].item()
    
    # Preparar resultados
    results = {
        'image_path': image_path,
        'original_size': original_size,
        'predicted_class': class_names[predicted_class],
        'predicted_class_idx': predicted_class,
        'confidence': confidence,
        'all_probabilities': {
            class_names[i]: float(probabilities[0, i])
            for i in range(len(class_names))
        }
    }
    
    # Log dos resultados
    logger.info(f"\nPredição para: {os.path.basename(image_path)}")
    logger.info(f"Classe predita: {results['predicted_class']} (confiança: {confidence:.4f})")
    logger.info("Todas as probabilidades:")
    for class_name, prob in results['all_probabilities'].items():
        logger.info(f"  {class_name}: {prob:.4f}")
    
    return results

def analyze_misclassifications(model: nn.Module, test_loader: DataLoader, 
                              class_names: List[str], device: str, 
                              num_examples: int = 10) -> List[Dict]:
    """Análise detalhada de classificações incorretas"""
    
    logger.info(f"Analisando classificações incorretas (top {num_examples})...")
    
    model.eval()
    misclassifications = []
    
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probabilities = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            # Encontrar classificações incorretas
            incorrect_mask = predicted != labels
            if incorrect_mask.sum() > 0:
                incorrect_indices = torch.where(incorrect_mask)[0]
                
                for idx in incorrect_indices:
                    true_label = labels[idx].item()
                    pred_label = predicted[idx].item()
                    confidence = probabilities[idx, pred_label].item()
                    true_confidence = probabilities[idx, true_label].item()
                    
                    misclass_info = {
                        'batch_idx': batch_idx,
                        'sample_idx': idx.item(),
                        'true_class': class_names[true_label],
                        'predicted_class': class_names[pred_label],
                        'confidence': confidence,
                        'true_class_confidence': true_confidence,
                        'confidence_gap': confidence - true_confidence,
                        'image_tensor': images[idx].cpu()
                    }
                    
                    misclassifications.append(misclass_info)
                    
                    if len(misclassifications) >= num_examples:
                        break
            
            if len(misclassifications) >= num_examples:
                break
    
    # Ordenar por gap de confiança (casos mais "confiantes" mas errados)
    misclassifications.sort(key=lambda x: x['confidence_gap'], reverse=True)
    
    # Log dos resultados
    logger.info(f"\nTop {len(misclassifications)} classificações incorretas:")
    for i, misclass in enumerate(misclassifications[:5]):  # Mostrar apenas top 5
        logger.info(f"{i+1}. Verdadeiro: {misclass['true_class']}, "
                   f"Predito: {misclass['predicted_class']}, "
                   f"Confiança: {misclass['confidence']:.4f}")
    
    return misclassifications

def example_prediction():
    """Exemplo de uso da função de predição"""
    
    # Configurar modelo para predição
    class_names = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']
    
    # Criar modelo
    model = OptimizedCOVIDClassifier(num_classes=len(class_names), 
                                   architecture='resnet50')
    model = model.to(device)
    
    # Carregar melhor modelo (se existir)
    try:
        checkpoint = load_best_model(model)
        logger.info("Modelo carregado com sucesso!")
    except FileNotFoundError:
        logger.warning("Modelo não encontrado. Use um modelo treinado para predições.")
        return
    
    # Exemplo de predição (substitua pelo caminho real da imagem)
    image_path = "exemplo_imagem.jpg"  # Substitua por uma imagem real
    
    if os.path.exists(image_path):
        try:
            results = predict_single_image(model, image_path, class_names, device)
            logger.info("Predição realizada com sucesso!")
        except Exception as e:
            logger.error(f"Erro na predição: {e}")
    else:
        logger.info("Para testar a predição, coloque uma imagem no caminho especificado.")

def main():
    """Função principal otimizada"""
    
    logger.info("="*80)
    logger.info("CLASSIFICADOR COVID-19 OTIMIZADO")
    logger.info("="*80)
    
    # Configurações
    config = {
        'dataset_root': './covid_dataset',  # Ajuste o caminho
        'batch_size': 32,
        'num_epochs': 50,
        'learning_rate': 0.001,
        'weight_decay': 1e-4,
        'dropout_rate': 0.3,
        'architecture': 'resnet50',  # ou 'efficientnet_b3'
        'early_stopping_patience': 10,
        'use_amp': True,
        'cache_images': False,  # True para datasets pequenos
        'num_workers': 4
    }
    
    class_names = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']
    
    try:
        # 1. Criar datasets
        logger.info("Criando datasets...")
        
        train_transform = get_optimized_transforms('train', use_albumentations=True)
        val_transform = get_optimized_transforms('val', use_albumentations=True)
        
        train_dataset = OptimizedCOVIDDataset(
            dataset_root=config['dataset_root'],
            split='train',
            transform=train_transform,
            cache_images=config['cache_images']
        )
        
        val_dataset = OptimizedCOVIDDataset(
            dataset_root=config['dataset_root'],
            split='val',
            transform=val_transform,
            cache_images=config['cache_images']
        )
        
        test_dataset = OptimizedCOVIDDataset(
            dataset_root=config['dataset_root'],
            split='test',
            transform=val_transform,
            cache_images=config['cache_images']
        )
        
        # 2. Criar DataLoaders
        logger.info("Criando DataLoaders...")
        
        train_loader = create_optimized_dataloader(
            train_dataset, config['batch_size'], is_train=True, 
            num_workers=config['num_workers']
        )
        
        val_loader = create_optimized_dataloader(
            val_dataset, config['batch_size'], is_train=False,
            num_workers=config['num_workers']
        )
        
        test_loader = create_optimized_dataloader(
            test_dataset, config['batch_size'], is_train=False,
            num_workers=config['num_workers']
        )
        
        # 3. Criar modelo
        logger.info("Criando modelo...")
        
        model = OptimizedCOVIDClassifier(
            num_classes=len(class_names),
            pretrained=True,
            dropout_rate=config['dropout_rate'],
            architecture=config['architecture']
        )
        model = model.to(device)
        
        # 4. Configurar treinamento
        logger.info("Configurando treinamento...")
        
        # Loss com pesos de classe
        class_weights = train_dataset.get_class_weights().to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        
        # Optimizer
        optimizer = optim.AdamW(
            model.parameters(),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay']
        )
        
        # Scheduler
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5, verbose=True
        )
        
        # 5. Criar trainer
        trainer = OptimizedModelTrainer(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            class_names=class_names,
            use_amp=config['use_amp']
        )
        
        # 6. Treinar modelo
        logger.info("Iniciando treinamento...")
        best_val_acc = trainer.train(
            num_epochs=config['num_epochs'],
            early_stopping_patience=config['early_stopping_patience']
        )
        
        # 7. Plotar histórico
        trainer.plot_training_history()
        
        # 8. Carregar melhor modelo e avaliar
        logger.info("Avaliando modelo no conjunto de teste...")
        
        checkpoint = load_best_model(model)
        results = evaluate_model_optimized(model, test_loader, class_names, device)
        
        # 9. Salvar informações
        model_info = save_model_info(model, trainer.history, results)
        
        # 10. Análise de erros
        misclassifications = analyze_misclassifications(
            model, test_loader, class_names, device, num_examples=20
        )
        
        logger.info("="*80)
        logger.info("TREINAMENTO CONCLUÍDO COM SUCESSO!")
        logger.info(f"Melhor acurácia de validação: {best_val_acc:.4f}")
        logger.info(f"Acurácia no teste: {results['accuracy']:.4f}")
        logger.info("="*80)
        
    except Exception as e:
        logger.error(f"Erro durante execução: {e}")
        raise

if __name__ == "__main__":
    main()

2025-06-29 12:19:33,697 - INFO - Dispositivo utilizado: cpu
2025-06-29 12:19:33,708 - INFO - ================================================================================
2025-06-29 12:19:33,709 - INFO - CLASSIFICADOR COVID-19 OTIMIZADO
2025-06-29 12:19:33,709 - INFO - ================================================================================
2025-06-29 12:19:33,710 - INFO - Criando datasets...
2025-06-29 12:19:33,711 - ERROR - Erro durante execução: 1 validation error for InitSchema
size
  Input should be a valid tuple [type=tuple_type, input_value=224, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/tuple_type


ValueError: 1 validation error for InitSchema
size
  Input should be a valid tuple [type=tuple_type, input_value=224, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/tuple_type